In [2]:
!nvidia-smi

Mon Sep  1 18:51:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.16                 Driver Version: 572.16         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   44C    P8              4W /  220W |    1143MiB /  12282MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Yolo v8 model

In [3]:
!pip install ultralytics


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: C:\Users\jakub\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
from ultralytics import YOLO

Podzielenie zestawu danych

In [5]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
!python split_data.py --datapath="./data" --train_pct=0.8

Created folder at D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data/train/images.
Created folder at D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data/train/labels.
Created folder at D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data/validation/images.
Created folder at D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data/validation/labels.
Number of image files: 151
Number of annotation files: 151
Images moving to train: 120
Images moving to validation: 31


In [7]:
import yaml
import os

In [8]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
    if not os.path.exists(path_to_classes_txt):
        print("clases.txt file not found!")
        return
    with open(path_to_classes_txt) as f:
        classes = []
        for line in f.readlines():
            if len(line.strip()) == 0: continue
            classes.append(line.strip())
        number_of_classes = len(classes)

        data = {
            'path':'data',
            'train':'./train/images',
            'val':'./validation/images',
            'nc': number_of_classes,
            'names': classes
        }

        with open(path_to_data_yaml, 'w') as f:
            yaml.dump(data, f, sort_keys=False)
        print("Created file")
        return
path_to_classes_txt = './data/classes.txt'
path_to_data_yaml = './data/data.yaml'
create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print("Created file")
!cat ./data/data.yaml

Created file
Created file


'cat' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
!yolo detect train data=./data/data.yaml model=yolo12s.pt epochs=60 imgsz=640

New https://pypi.org/project/ultralytics/8.3.191 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.139  Python-3.10.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train14, nbs=64, nms=False, opset=None, opti


  0%|          | 0.00/5.35M [00:00<?, ?B/s]
 33%|###2      | 1.75M/5.35M [00:00<00:00, 17.3MB/s]
 65%|######5   | 3.50M/5.35M [00:00<00:00, 8.34MB/s]
 86%|########6 | 4.62M/5.35M [00:00<00:00, 6.82MB/s]
100%|##########| 5.35M/5.35M [00:00<00:00, 7.92MB/s]

train: Scanning D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\train\labels...:   0%|          | 0/120 [00:00<?, ?it/s]
train: Scanning D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\train\labels... 2 images, 0 backgrounds, 0 corrupt:   2%|1         | 2/120 [00:00<00:06, 17.24it/s]
train: Scanning D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\train\labels... 4 images, 0 backgrounds, 0 corrupt:   3%|3         | 4/120 [00:00<00:07, 15.27it/s]
train: Scanning D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\train\labels... 25 images, 0 backgrounds, 0 corrupt:  21%|##        | 25/120 [00:00<00:01, 52.49it/s]
train: Scanning D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\train\labels... 51 images, 0 backgrounds, 0 corrupt:  42%|####2     | 51/120 [00:00<00:0

In [10]:
path_to_model = './znaczki_12s/weights/best.pt'
!yolo detect predict model=./znaczki_12s/weights/best.pt source=data/validation/images save=True

Ultralytics 8.3.139  Python-3.10.11 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLOv12s summary (fused): 159 layers, 9,252,552 parameters, 0 gradients, 21.3 GFLOPs

image 1/31 D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\validation\images\19f30e7b-IMG_4244.JPG: 640x352 1 1, 1 miecz, 1 wiez, 41.5ms
image 2/31 D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\validation\images\25ada29f-Skan_20250728-22124.jpg: 640x352 1 pozoga, 9.2ms
image 3/31 D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\validation\images\368004d6-Skan_20250728-2204.jpg: 640x352 1 6, 1 miecz, 8.6ms
image 4/31 D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\validation\images\3703ba57-Skan_20250728-22121.jpg: 640x352 1 4, 1 miecz, 1 szpiegostwo, 6.8ms
image 5/31 D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\validation\images\39402de2-Skan_20250728-22152.jpg: 640x352 1 1, 1 lucznik, 1 zmartwychwstanie, 6.7ms
image 6/31 D:\pliki\Studia\kolo\ML_GWINT_PROJECT\data\validation\images\3ce79fbd-IMG_4225.JPG: 640x352 1 7, 1 b

In [11]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(path_to_model + '/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

In [12]:
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train
%cd /content

The syntax of the command is incorrect.
'cp' is not recognized as an internal or external command,
operable program or batch file.


[WinError 2] Nie można odnaleźć określonego pliku: 'my_model'
D:\pliki\Studia\kolo\ML_GWINT_PROJECT


'cp' is not recognized as an internal or external command,
operable program or batch file.
C:\Users\jakub\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
'zip' is not recognized as an internal or external command,
operable program or batch file.


[WinError 2] Nie można odnaleźć określonego pliku: '/content'
D:\pliki\Studia\kolo\ML_GWINT_PROJECT


'zip' is not recognized as an internal or external command,
operable program or batch file.
